In [1]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [2]:
url_t1 = "https://www.data.gouv.fr/api/1/datasets/r/4feeef01-24f7-4d5a-914f-8aa806f31ec2"
t1_muni_26 = pd.read_csv(url_t1, sep=";")
t1_muni_26.head()

URLError: <urlopen error [Errno 111] Connection refused>

In [3]:
FILE_KEY_OUT_S3 = "t1_muni.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    t1_muni_26.to_csv(file_out)

PermissionError: Access Denied.

In [4]:
url_t2 = "https://www.data.gouv.fr/api/1/datasets/r/6ff67a28-01bf-459e-beca-dd7aa8132dc1"
t2_muni_26 = pd.read_csv(url_t2, sep=";")
t2_muni_26.head()

URLError: <urlopen error [Errno 111] Connection refused>

In [5]:
FILE_KEY_OUT_S3 = "t2_muni.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    t2_muni_26.to_csv(file_out)

PermissionError: Access Denied.

In [ ]:
# On récupère dans le répertoire les établissement dont l'activité principale est celle d'un débit de boissons (56.30Z)
url_sirene = "https://www.data.gouv.fr/api/1/datasets/r/a29c1297-1f92-4e2a-8f6b-8c902ce96c5f"
tous_bistrots_sirene = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{url_sirene}')
    WHERE activitePrincipaleEtablissement == '56.30Z'
""").df()
tous_bistrots_sirene.head()

# Sur la table hors filtre état admin la répartition est la suivante : 
# # etatAdministratifEtablissement
#F    83401
#A    50266

# Note : La variable du code commune s'appelle ici codeCommuneEtablissement. (Il y a deux colonnes supplémentaires appendées 2 et 3, je suppose qu'il s'agit d'établissements ayant déménagé mais on peut l'ignorer dans un premier temps)

In [ ]:
url_sirene_historique = "https://www.data.gouv.fr/api/1/datasets/r/2b3a0c79-f97b-46b8-ac02-8be6c1f01a8c"
bistrots_sirene_fermetures = duckdb.sql(f"""
    SELECT siret, dateDebut AS date_fermeture
    FROM read_parquet('{url_sirene_historique}')
    WHERE etatAdministratifEtablissement = 'F'
      AND changementEtatAdministratifEtablissement = 'true'
      AND activitePrincipaleEtablissement == '56.30Z'
""").df()

In [ ]:
bistrots_sirene_actifs = tous_bistrots_sirene.merge(
  bistrots_sirene_fermetures,
  left_on = ["siret"],
  right_on = ["siret"],
  how = "left"
)

In [ ]:
bistrots_sirene = bistrots_sirene_actifs[((bistrots_sirene_actifs.date_fermeture >= "2026-01-04") | (bistrots_sirene_actifs.date_fermeture.isnull())) & (bistrots_sirene_actifs.dateCreationEtablissement <= "2026-03-01")]

In [ ]:
bistrots_sirene.etatAdministratifEtablissement.value_counts()
# Tous les bistrots retenus sont en statut ouvert (au 30 septembre 2026), donc on a bien supprimé ceux qui ont fermé dans l'intervalle. 

In [ ]:
FILE_KEY_OUT_S3 = "bistrots_sirene.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    bistrots_sirene.to_csv(file_out)

In [ ]:
#Récupération des données Filosofi au niveau commune

def get_melodi(dataset, dimension, code, value_name, time_period, geo_level="COM"):

  url = f"https://api.insee.fr/melodi/data/{dataset}"

  params = {
    "GEO": geo_level,
    dimension: code,
    "TIME_PERIOD": time_period,
    "maxResult": 40000,
  }

  response = requests.get(url, params=params, timeout=60)
  response.raise_for_status()
  observations = response.json()["observations"]

  return pd.DataFrame(
    {
      "CODGEO": obs["dimensions"]["GEO"].split("-")[-1],
      value_name: obs["measures"]["OBS_VALUE_NIVEAU"].get("value"),
    }
    for obs in observations
  )

niveau_vie = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "MED_SL", "NIVVIE_MEDIAN", 2023)
taux_pauvrete = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "PR_MD60", "TAUX_PAUVRETE", 2023)
population = get_melodi("DS_POPULATIONS_HISTORIQUES", "POPREF_MEASURE", "PMUN", "POPULATION", 2023)

url_cog_2023 = "https://www.insee.fr/fr/statistiques/fichier/6800675/v_commune_2023.csv"
url_backup = "https://minio.lab.sspcloud.fr/lgaliana/data/python-ENSAE/cog_2023.csv"

try:
  cog_2023 = pd.read_csv(url_cog_2023)
except requests.exceptions.Timeout:
  cog_2023 = pd.read_csv(url_backup)

noms_communes = (
  cog_2023
  .loc[cog_2023["TYPECOM"] == "COM", ["COM", "LIBELLE"]]
  .rename(columns={"COM": "CODGEO", "LIBELLE": "LIBGEO"})
)

filosofi = (
  niveau_vie
  .merge(taux_pauvrete, on="CODGEO", how="outer")
  .merge(population, on="CODGEO", how="outer")
  .merge(noms_communes, on="CODGEO", how="left")
  [["CODGEO", "LIBGEO", "POPULATION", "NIVVIE_MEDIAN", "TAUX_PAUVRETE"]]
)

filosofi["POPULATION"] = filosofi["POPULATION"].astype(int)
filosofi["dep"] = filosofi["CODGEO"].str[:2]

In [ ]:
filosofi.head()

In [ ]:
FILE_KEY_OUT_S3 = "filosofi.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    filosofi.to_csv(file_out)

In [ ]:
fs.ls("mathildedonnenfeld/diffusion/bistrots_et_bulletins/")